# Process tree graph

- Build a parent-child graph from process creation events
- Find the ancestry of any process
- Draw the tree under a suspicious process

---
<sub>TechDetechtives analytics workbench. This notebook is a modified work derived from the HELK notebook `tutorials/06-Intro_pyspark_graphframes_sysmon.ipynb` (https://github.com/Cyb3rWard0g/HELK), Copyright Roberto Rodriguez (@Cyb3rWard0g), licensed GPL-3.0. Modified by TechDetechtives, 2026: builds the parent-child graph with NetworkX instead of GraphFrames, reads from the platform through a read-only client, and uses ECS field names. Licensed GPL-3.0; see `analytics/LICENSE`.</sub>

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import td_hunt as td

## Build the graph

Each node is one process instance. An edge points from a parent to the process it started.

In [ ]:
proc = td.events("process", since="24h")
graph = td.process_tree(proc)
print(graph.number_of_nodes(), "processes,", graph.number_of_edges(), "parent-child links")

## Deepest chains

Long chains of processes starting processes are unusual for ordinary desktop work.

In [ ]:
leaves = [n for n in graph if graph.out_degree(n) == 0]
chains = sorted((td.lineage(graph, leaf) for leaf in leaves), key=len, reverse=True)
for chain in chains[:5]:
    print(" > ".join(chain))

## Everything started under one process

Pick the script interpreters started by the WMI provider host and collect all of their descendants.

In [ ]:
roots = [
    n for n in graph
    if (graph.nodes[n].get("name") or "").lower() in ("powershell.exe", "pwsh.exe", "cmd.exe")
    and any((graph.nodes[p].get("name") or "").lower() == "wmiprvse.exe" for p in graph.predecessors(n))
]
for root in roots:
    data = graph.nodes[root]
    print(f"{data['host']} | {data['user']} | {data['name']}")
    for child in nx.descendants(graph, root):
        print("   ", graph.nodes[child]["name"], "|", graph.nodes[child]["command_line"])

## Draw the tree

In [ ]:
def draw_tree(graph, root):
    ancestors = nx.ancestors(graph, root)
    sub = graph.subgraph(ancestors | {root} | nx.descendants(graph, root))
    # Lay nodes out by depth: one row per generation.
    top = [n for n in sub if sub.in_degree(n) == 0][0]
    depth = nx.single_source_shortest_path_length(sub, top)
    rows = {}
    for node, level in depth.items():
        rows.setdefault(level, []).append(node)
    pos = {
        node: ((i + 1) / (len(nodes) + 1), -level)
        for level, nodes in rows.items()
        for i, node in enumerate(sorted(nodes))
    }
    labels = {n: sub.nodes[n].get("name") or "?" for n in sub}
    colors = ["#d9534f" if n == root else "#5b8def" for n in sub]
    fig, ax = plt.subplots(figsize=(9, 1.3 * (max(rows) + 1) + 1))
    nx.draw_networkx(sub, pos, labels=labels, node_color=colors, node_size=1800,
                     font_size=8, arrows=True, ax=ax)
    ax.set_axis_off()
    ax.margins(0.15)
    plt.tight_layout()
    return fig

if roots:
    draw_tree(graph, roots[0])
else:
    print("No script interpreter started by the WMI provider host in this window.")